In [5]:
import pandas as pd


#read in data

url = 'https://redfin-public-data.s3.us-west-2.amazonaws.com/redfin_data_center/housing_market/weekly/all_metros.csv'
raw_data = pd.read_csv(url)



In [6]:
# explore data and shape and columns

print(raw_data.head())
print(raw_data.shape)

print(raw_data.columns.tolist())

# confirm it is metro data
print(raw_data['REGION TYPE'].unique())



  LAST UPDATED FREQUENCY PERIOD BEGIN  PERIOD END  REGION ID REGION TYPE  \
0   2026-09-15    Weekly   2026-08-17  2026-09-13      10100       Metro   
1   2026-09-15    Weekly   2026-08-17  2026-09-13      10140       Metro   
2   2026-09-15    Weekly   2026-08-17  2026-09-13      10180       Metro   
3   2026-09-15    Weekly   2026-08-17  2026-09-13      10220       Metro   
4   2026-09-15    Weekly   2026-08-17  2026-09-13      10300       Metro   

               REGION NAME  HOMES SOLD NSA  HOMES SOLD NSA YOY (%)  \
0  Aberdeen, SD metro area            43.0                   16.48   
1  Aberdeen, WA metro area            91.0                    5.12   
2   Abilene, TX metro area           200.0                   -4.44   
3       Ada, OK metro area            34.0                  -13.75   
4    Adrian, MI metro area           110.0                    0.34   

   MEDIAN SALE PRICE NSA ($)  ...  PENDING SALES SA WOW (%)  \
0                   310931.0  ...                       NaN

In [7]:
# count number of regions
print(raw_data['REGION ID'].nunique())

946


In [24]:
#convert period start/end into datetime objects
raw_data['PERIOD BEGIN'] = pd.to_datetime(raw_data['PERIOD BEGIN'])
raw_data['PERIOD END'] = pd.to_datetime(raw_data['PERIOD END'])

#ensure that all period lengths are 27 days
period_length = (raw_data['PERIOD END'] - raw_data['PERIOD BEGIN']).dt.days

print(period_length.unique())

[27]


In [8]:
# rename columns so it can read into BigQuery
raw_data.columns = (
    raw_data.columns
    .str.strip() # strip whitespace
    .str.lower() # lowercase all column names
    .str.replace(r'[^\w\s]', '', regex=True)   # remove $, %, (), etc.
    .str.replace(r'\s+', '_', regex=True)       # replace spaces with underscores
)

print(raw_data.columns.tolist())

['last_updated', 'frequency', 'period_begin', 'period_end', 'region_id', 'region_type', 'region_name', 'homes_sold_nsa', 'homes_sold_nsa_yoy_', 'median_sale_price_nsa_', 'median_sale_price_nsa_yoy_', 'median_days_on_market_nsa_days', 'median_days_on_market_nsa_yoy_', 'average_sale_to_list_ratio_nsa_', 'average_sale_to_list_ratio_nsa_yoy_ppts', 'share_sold_above_original_list_nsa_', 'share_sold_above_original_list_nsa_yoy_ppts', 'median_sale_price_per_sqft_nsa_', 'median_sale_price_per_sqft_nsa_yoy_', 'months_of_supply_nsa', 'months_of_supply_nsa_yoy_', 'percent_off_market_in_two_weeks_nsa_', 'percent_off_market_in_two_weeks_nsa_yoy_ppts', 'new_listings_sa', 'new_listings_nsa', 'new_listings_sa_wow_', 'new_listings_sa_yoy_', 'new_listings_nsa_yoy_', 'active_listings_sa', 'active_listings_nsa', 'active_listings_sa_wow_', 'active_listings_sa_yoy_', 'active_listings_nsa_yoy_', 'pending_sales_sa', 'pending_sales_nsa', 'pending_sales_sa_wow_', 'pending_sales_sa_yoy_', 'pending_sales_nsa_yoy_

In [9]:
# save data to csv

raw_data.to_csv('redfin_all_metros.csv', index=False)

## Infrastructure setup

The cells below create the BigQuery client, dataset, and raw table used
throughout this project. This logic was later migrated into the Airflow DAG
(`airflow/dags/redfin_pipeline.py`) to run as part of the automated pipeline;
it's kept here as a record of the initial manual setup and validation.

In [10]:
from google.cloud import bigquery

client = bigquery.Client()
#print(client.project)

In [11]:
# create dataset

dataset_id = f"{client.project}.redfin_raw"
dataset = bigquery.Dataset(dataset_id)
dataset.location = "US"

dataset = client.create_dataset(dataset, exists_ok=True)
print(f"Created dataset {dataset.dataset_id}")

Created dataset redfin_raw


In [12]:
#save csv into a raw table inside the dataset


table_id = f"{client.project}.redfin_raw.metro_weekly_housing"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,       # skips the header row
    autodetect=True,           # let BigQuery guess types for now
    write_disposition="WRITE_TRUNCATE",  # overwrite table if it already exists
)

with open("redfin_all_metros.csv", "rb") as source_file:
    load_job = client.load_table_from_file(source_file, table_id, job_config=job_config)

load_job.result()  # waits for the job to finish

table = client.get_table(table_id)
print(f"Loaded {table.num_rows} rows into {table_id}")
print("Schema:")
for field in table.schema:
    print(f"  {field.name}: {field.field_type}")

Loaded 365970 rows into project-93ae46fe-b485-4962-b2c.redfin_raw.metro_weekly_housing
Schema:
  last_updated: DATE
  frequency: STRING
  period_begin: DATE
  period_end: DATE
  region_id: INTEGER
  region_type: STRING
  region_name: STRING
  homes_sold_nsa: FLOAT
  homes_sold_nsa_yoy_: FLOAT
  median_sale_price_nsa_: FLOAT
  median_sale_price_nsa_yoy_: FLOAT
  median_days_on_market_nsa_days: FLOAT
  median_days_on_market_nsa_yoy_: FLOAT
  average_sale_to_list_ratio_nsa_: FLOAT
  average_sale_to_list_ratio_nsa_yoy_ppts: FLOAT
  share_sold_above_original_list_nsa_: FLOAT
  share_sold_above_original_list_nsa_yoy_ppts: FLOAT
  median_sale_price_per_sqft_nsa_: FLOAT
  median_sale_price_per_sqft_nsa_yoy_: FLOAT
  months_of_supply_nsa: FLOAT
  months_of_supply_nsa_yoy_: FLOAT
  percent_off_market_in_two_weeks_nsa_: FLOAT
  percent_off_market_in_two_weeks_nsa_yoy_ppts: FLOAT
  new_listings_sa: FLOAT
  new_listings_nsa: FLOAT
  new_listings_sa_wow_: FLOAT
  new_listings_sa_yoy_: FLOAT
  new_li